# P43 — Normalización por lotes: acelerar el entrenamiento profundo

## 1. Título y paper

**Paper:** *Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate Shift*  
**Autoría:** Sergey Ioffe, Christian Szegedy  
**Año y venue:** 2015 · arXiv:1502.03167 · ICML 2015  
**Nivel:** L2 · **Motor:** `batchnorm`  
**Ficha completa:** [`P43_batchnorm`](../../papers/foundational/P43_batchnorm/README.md)

**Hito:** Normalizar las activaciones dentro de la red permite tasas de aprendizaje mucho mayores y hace el entrenamiento profundo mucho menos frágil.

- [arXiv:1502.03167](https://arxiv.org/abs/1502.03167)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Entrenar redes profundas exigía inicializaciones cuidadosas y tasas de aprendizaje pequeñas: la distribución de las activaciones de cada capa se desplazaba durante el entrenamiento.
2. Ejecutar una implementación mínima de la propuesta: Normalizar cada activación usando la media y la varianza del minilote, y añadir dos parámetros aprendidos (γ, β) para que la red pueda deshacer la normalización si le conviene.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P02
- P04


## 4. Intuición

Una tubería de doce filtros donde cada uno amplifica un poco. Al final, la señal está saturada o extinguida. Normalizar entre etapas es reajustar el caudal para que cada filtro reciba lo que sabe procesar.


## 5. Concepto mínimo

```text
x̂ = (x − μ_lote) / √(σ²_lote + ε)
y  = γ·x̂ + β                    γ y β se APRENDEN
```

γ y β permiten deshacer la normalización si conviene: no se pierde capacidad expresiva.


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('batchnorm', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Qué fracción de activaciones estará saturada en la capa 11 sin normalizar?
2. ¿Y con normalización?
3. ¿Por qué importa la saturación de tanh?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('batchnorm', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('batchnorm', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

Sin normalizar, casi todas las activaciones acaban en la zona plana de tanh, donde la derivada es prácticamente cero: **el gradiente no puede volver**. Con normalización, la desviación se mantiene cerca de 1 y las unidades siguen en su rango útil.


## 10. Comentario pedagógico

La explicación del paper —«internal covariate shift»— fue **discutida después**. Hay evidencia de que el beneficio real viene de suavizar el paisaje de optimización. Es un caso didáctico excelente: la técnica funciona y su explicación original resultó incompleta.


## 11. Error o anti-patrón deliberado

Anti-patrón: usar batch norm con lotes muy pequeños.


In [ ]:
for lote in (1, 2, 8, 64, 256):
    error = 1 / (lote ** 0.5)
    print(f'lote {lote:>3} → error relativo de la estadistica ~{error:.2f} '
          f"({'inutilizable' if lote < 8 else 'aceptable'})")

## 12. Corrección

Por eso existen LayerNorm y GroupNorm, que no dependen del lote:


In [ ]:
variantes = {'BatchNorm': 'normaliza por LOTE — depende del tamano de lote',
             'LayerNorm': 'normaliza por MUESTRA — la que usa el Transformer',
             'GroupNorm': 'por grupos de canales — para lotes pequenos en vision'}
show(variantes)

## 13. Desafío guiado

Comprueba qué pasa con la desviación en la capa 11 si subes el peso de 1,6 a 2,5.


In [ ]:
r = run_paper_lab('batchnorm', seed=3)['result']
show(r)

## 14. Desafío autónomo

Entrena una red profunda con y sin normalización, variando la tasa de aprendizaje en un rango amplio. Reporta el rango de tasas que converge en cada caso: ahí está el beneficio real.


## 15. Evidencia de aprendizaje

Guarda las trazas de media, desviación y saturación por capa, y tu comparación de las tres variantes de normalización.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P43_batchnorm/README.md) · evaluación formal: [`assessments/papers/P43_batchnorm.md`](../../assessments/papers/P43_batchnorm.md)


## 16. Cierre

Con normalización se entrena más profundo. Pero pasado cierto punto, más capas volvían a empeorar — y no por sobreajuste.


## 17. Conexión con el siguiente hito

- P44
- LayerNorm y P08

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
